In [ ]:
import os

import numpy as np
import torch

from rect_prune_ood.lib.util.paths import get_top_level_path

results_dir = os.path.join(get_top_level_path(), 'results')
imagenet_c_results_dir = os.path.join(results_dir, 'imagenet', 'baseline', 'imagenet_c')

In [ ]:
imagenet_c_activations = np.load(
	os.path.join(imagenet_c_results_dir, 'imagenet_c_activations_severity_5.npy')
)

imagenet_c_perturbations = [
	'brightness',
	'defocus_blur',
	'fog',
	'gaussian_blur',
	'glass_blur',
	'jpeg_compression',
	'pixelate',
	'shot_noise',
	'spatter',
	'zoom_blur',
	'contrast',
	'elastic_transform',
	'frost',
	'gaussian_noise',
	'impulse_noise',
	'motion_blur',
	'saturate',
	'snow',
	'speckle_noise',
]

In [7]:
percentile = 95

mask_dict = {}

for perturbation in range(len(imagenet_c_perturbations)):
	activations = imagenet_c_activations[perturbation]
	variance = np.var(activations, axis=1)
	variance = torch.tensor(variance)

	p = np.percentile(variance, percentile)

	mask = torch.where(variance <= p, 1, 0)
	mask = mask.view(variance.shape[0], 1, 1)

	mask_dict[imagenet_c_perturbations[perturbation]] = mask

In [15]:
# Find masks with greatest euclidean distance to each other
max_distance = 0
max_mask_pair = None
mask_pair_distances = {}
for i, (k1, v1) in enumerate(mask_dict.items()):
	for j, (k2, v2) in enumerate(mask_dict.items()):
		if i >= j:
			continue

		distance = torch.dist(v1.float(), v2.float())
		mask_pair_distances[(k1, k2)] = distance
		if distance > max_distance:
			max_distance = distance
			max_mask_pair = (k1, k2)

In [16]:
mask_pair_distances_sorted = {
	k: v
	for k, v in sorted(
		mask_pair_distances.items(), key=lambda item: item[1], reverse=True
	)
}
mask_pair_distances_sorted

{('frost', 'impulse_noise'): tensor(11.9164),
 ('fog', 'impulse_noise'): tensor(11.8322),
 ('zoom_blur', 'impulse_noise'): tensor(11.8322),
 ('contrast', 'impulse_noise'): tensor(11.8322),
 ('impulse_noise', 'saturate'): tensor(11.6619),
 ('pixelate', 'impulse_noise'): tensor(11.5758),
 ('defocus_blur', 'impulse_noise'): tensor(11.4891),
 ('zoom_blur', 'snow'): tensor(11.4018),
 ('brightness', 'impulse_noise'): tensor(11.3137),
 ('gaussian_noise', 'snow'): tensor(11.3137),
 ('fog', 'shot_noise'): tensor(11.2250),
 ('gaussian_blur', 'impulse_noise'): tensor(11.2250),
 ('shot_noise', 'snow'): tensor(11.2250),
 ('impulse_noise', 'motion_blur'): tensor(11.2250),
 ('jpeg_compression', 'impulse_noise'): tensor(11.1355),
 ('spatter', 'impulse_noise'): tensor(11.1355),
 ('elastic_transform', 'impulse_noise'): tensor(11.1355),
 ('fog', 'gaussian_noise'): tensor(11.0454),
 ('glass_blur', 'impulse_noise'): tensor(11.0454),
 ('shot_noise', 'zoom_blur'): tensor(11.0454),
 ('impulse_noise', 'snow'):

In [17]:
# Find mask with greatest average distance to all other masks
average_distances = {}
for k1, v1 in mask_dict.items():
	distances = []
	for k2, v2 in mask_dict.items():
		if k1 == k2:
			continue
		distance = torch.dist(v1.float(), v2.float())
		distances.append(distance)
	average_distances[k1] = np.mean(distances)

average_distances_sorted = {
	k: v
	for k, v in sorted(
		average_distances.items(), key=lambda item: item[1], reverse=True
	)
}
average_distances_sorted

{'impulse_noise': 11.17177,
 'snow': 10.400344,
 'fog': 9.901767,
 'frost': 9.836271,
 'zoom_blur': 9.83099,
 'shot_noise': 9.801041,
 'gaussian_noise': 9.656132,
 'saturate': 9.430584,
 'contrast': 9.398479,
 'speckle_noise': 9.361217,
 'glass_blur': 9.142362,
 'pixelate': 8.7769375,
 'defocus_blur': 8.731296,
 'motion_blur': 8.626728,
 'spatter': 8.486869,
 'jpeg_compression': 8.395288,
 'brightness': 8.179845,
 'elastic_transform': 8.139332,
 'gaussian_blur': 8.128221}